In [45]:
import json

In [47]:
import json
import re

# Load JSON
with open("../../data_augmentatio_stefano/mitre/object_to_ttp_links_filled.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Regex that removes "(Citation: ...)"
citation_pattern = re.compile(r"\(Citation:.*?\)")

for entry in data:
    desc = entry.get("relationship_description", "")
    # Remove citation(s) and clean extra spaces
    cleaned = citation_pattern.sub("", desc).strip()
    # Remove double spaces created by deletion
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    entry["relationship_description"] = cleaned

# Save output
with open("../../data_augmentatio_stefano/mitre/cleaned.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)


In [1]:
example = {
    "sentence": "The actor used PowerShell to download and execute payloads.",
    "positive_ttps": ["T1059.001"],
}


In [3]:
ttp_texts = {
    "T1059.001": "Tactic: Execution\nID: T1059.001\nName: PowerShell\nDescription: Adversaries may abuse PowerShell...",
    "T1003.001": "..."
}


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader

model_name = "microsoft/deberta-v3-large"  # or similar
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1  # single logit for relevance
)

class TtpPairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs  # list of dicts: {"sentence":..., "ttp_id":..., "label":0/1}

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ex = self.pairs[idx]
        sent = ex["sentence"]
        ttp_id = ex["ttp_id"]
        label = ex["label"]

        ttp_text = ttp_texts[ttp_id]
        text = f"Sentence: {sent}\n\nTTP:\n{ttp_text}"

        enc = tokenizer(
            text,
            truncation=True,
            max_length=512,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),
        }

# build train_pairs = list of {sentence, ttp_id, label}
train_dataset = TtpPairDataset(train_pairs)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.BCEWithLogitsLoss()

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].cuda(),
            attention_mask=batch["attention_mask"].cuda()
        )
        logits = outputs.logits.squeeze(-1)  # (B,)
        loss = criterion(logits, batch["labels"].cuda())
        loss.backward()
        optimizer.step()


KeyboardInterrupt: 